In [1]:
import pandas as pd
import numpy as np

In [2]:
import os
from google.colab import drive
drive.mount('/content/drive')
# 특정 폴더를 기본 디렉토리로 변경
os.chdir("/content/drive/MyDrive/비어플/dataset")

# 현재 디렉토리 확인
print("현재 디렉토리:", os.getcwd())

Mounted at /content/drive
현재 디렉토리: /content/drive/MyDrive/비어플/dataset


In [4]:
import pandas as pd
import numpy as np

# 1. 데이터 불러오기
infra_df = pd.read_csv("infra_clustered_mean.csv")
weights_df = pd.read_csv("AHP_weights.csv").set_index("유형")

# 2. 인프라 항목을 인덱스로 설정하고 전치
infra_df = infra_df.rename(columns={"index": "infra_item"}).set_index("infra_item")
cluster_df = infra_df.T  # 클러스터가 행, 인프라 항목이 열

# 3. TOPSIS 함수 정의
def topsis(matrix: pd.DataFrame, weight: np.ndarray) -> pd.Series:
    values = matrix.select_dtypes(include=[np.number]).values
    norm_matrix = values / np.linalg.norm(values, axis=0)
    weighted_matrix = norm_matrix * weight

    ideal = weighted_matrix.max(axis=0)
    anti_ideal = weighted_matrix.min(axis=0)

    d_pos = np.linalg.norm(weighted_matrix - ideal, axis=1)
    d_neg = np.linalg.norm(weighted_matrix - anti_ideal, axis=1)

    return d_neg / (d_pos + d_neg)

# 4. 유형별 TOPSIS 점수 계산
result_df = pd.DataFrame({
    user_type: topsis(cluster_df, weights_df.loc[user_type].reindex(cluster_df.columns).values)
    for user_type in weights_df.index
}, index=cluster_df.index)

# 5. 결과 출력
print("TOPSIS 결과 (귀농 유형별 클러스터 선호도):")
result_df

TOPSIS 결과 (귀농 유형별 클러스터 선호도):


,은퇴생계형,청년창농형,가족정착형,영농상속형
cluster_0,0.821190,0.536177,0.719917,0.872344
cluster_1,0.000000,0.000000,0.000000,0.000000
cluster_2,0.328088,0.553964,0.346161,0.319041
cluster_3,0.509655,0.336034,0.311381,0.529229


In [12]:
# 숫자형 인덱스로 변경
result_df.index = result_df.index.str.replace("cluster_", "").astype(int)

type_to_cluster = result_df.idxmax().to_dict()

# 결과 저장
mapping_df = pd.DataFrame(list(type_to_cluster.items()), columns=["유형", "클러스터"])
mapping_df.to_csv("cluster_matching_result.csv", index=False)